In [1]:
import sys
PATH_TO_PREDICTOR_HOME = ".."
sys.path.append(PATH_TO_PREDICTOR_HOME)


from mhcnames import normalize_allele_name
import numpy as np
import pandas as pd
from config import RESULT_COLUMNS


In [2]:
DATA_PATH = "/home/denklewer/Data/gbm122/pVACseq/"


In [3]:
from data_reading import read_gbm122_data

per_region_total_df , per_region_per_allele_mt_df, per_region_per_allele_wt_df = read_gbm122_data(DATA_PATH)


Region 1, unique (Transcrpit, Epitope pairs): 13197
found 5 alleles: ['HLA-A*03:01' 'HLA-B*35:01' 'HLA-B*40:01' 'HLA-C*03:04' 'HLA-C*04:01']
allele HLA-A*03:01: initial length 13221
take unique (MT_epitope, WT_epitope) pairs
result length 5409
allele HLA-B*35:01: initial length 13221
take unique (MT_epitope, WT_epitope) pairs
result length 5409
allele HLA-B*40:01: initial length 13221
take unique (MT_epitope, WT_epitope) pairs
result length 5409
allele HLA-C*03:04: initial length 13221
take unique (MT_epitope, WT_epitope) pairs
result length 5409
allele HLA-C*04:01: initial length 13221
take unique (MT_epitope, WT_epitope) pairs
result length 5409
Region 2, unique (Transcrpit, Epitope pairs): 13273
found 5 alleles: ['HLA-A*03:01' 'HLA-B*35:01' 'HLA-B*40:01' 'HLA-C*03:04' 'HLA-C*04:01']
allele HLA-A*03:01: initial length 13279
take unique (MT_epitope, WT_epitope) pairs
result length 5127
allele HLA-B*35:01: initial length 13279
take unique (MT_epitope, WT_epitope) pairs
result length 51

/home/denklewer/Tools/anacoda3/envs/netmhc/lib/python3.9/site-packages/IPython/core/interactiveshell.py:3361: DtypeWarning: Columns (10) have mixed types.Specify dtype option on import or set low_memory=False.
  if (await self.run_code(code, result,  async_=asy)):


Region 3, unique (Transcrpit, Epitope pairs): 15170
found 5 alleles: ['HLA-A*03:01' 'HLA-B*35:01' 'HLA-B*40:01' 'HLA-C*03:04' 'HLA-C*04:01']
allele HLA-A*03:01: initial length 15170
take unique (MT_epitope, WT_epitope) pairs
result length 4971
allele HLA-B*35:01: initial length 15170
take unique (MT_epitope, WT_epitope) pairs
result length 4971
allele HLA-B*40:01: initial length 15170
take unique (MT_epitope, WT_epitope) pairs
result length 4971
allele HLA-C*03:04: initial length 15170
take unique (MT_epitope, WT_epitope) pairs
result length 4971
allele HLA-C*04:01: initial length 15170
take unique (MT_epitope, WT_epitope) pairs
result length 4971
Region 4, unique (Transcrpit, Epitope pairs): 13757
found 5 alleles: ['HLA-A*03:01' 'HLA-B*35:01' 'HLA-B*40:01' 'HLA-C*03:04' 'HLA-C*04:01']
allele HLA-A*03:01: initial length 13757
take unique (MT_epitope, WT_epitope) pairs
result length 4932
allele HLA-B*35:01: initial length 13757
take unique (MT_epitope, WT_epitope) pairs
result length 49

## Look at sample df

In [4]:
sample_region_dict = next(iter(per_region_per_allele_wt_df.values()))
sample_allele_df =  next(iter(sample_region_dict.values()))
sample_allele_df

,HLA Allele,Peptide Length,peptide,source_sequence_name
0,HLA-A*03:01,8,MYSDKRNK,seq0
1,HLA-A*03:01,8,NaN,seq1
2,HLA-A*03:01,8,NaN,seq2
3,HLA-A*03:01,8,AFYVGGSK,seq3
4,HLA-A*03:01,8,NaN,seq4
...,...,...,...,...
52720,HLA-A*03:01,11,PEPESEPEPEP,seq52720
52721,HLA-A*03:01,11,EPEPEPEPESE,seq52721
52722,HLA-A*03:01,11,EPEPEPESEPE,seq52722
52723,HLA-A*03:01,11,EPEPESEPEPE,seq52723


In [5]:
sample_allele_df['length'].min()

8

# 0. Preparations

In [6]:
ALLELES = list(sample_region_dict.keys())
REGIONS = list(per_region_total_df.keys())
ALLELES, REGIONS

(['HLA-A*03:01', 'HLA-B*35:01', 'HLA-B*40:01', 'HLA-C*03:04', 'HLA-C*04:01'],
 [1, 2, 3, 4, 5])

## 0.1. make result dicts

In [7]:
per_region_per_method_per_allele_wt_predictions = dict()
per_region_per_method_per_allele_mt_predictions = dict()
for region in per_region_total_df.keys():
    per_region_per_method_per_allele_wt_predictions[region] = dict()
    per_region_per_method_per_allele_mt_predictions[region] = dict()

## 0.2 Define TARGET_ALLELES (alleles than should be passed to algorithm)

In [9]:
TARGET_ALLELES = dict()

## HARDCODE alelle name which will be passed to algorithm
for allele_name in ALLELES:
    if allele_name == 'H2_d class II':
        target_allele_name = 'H2_IAd'
    elif allele_name == 'H2_b class II':
        target_allele_name = 'H2_IAb'
    else:
        target_allele_name = allele_name
    TARGET_ALLELES[allele_name] = (target_allele_name)


TARGET_ALLELES

{'HLA-A*03:01': 'HLA-A*03:01',
 'HLA-B*35:01': 'HLA-B*35:01',
 'HLA-B*40:01': 'HLA-B*40:01',
 'HLA-C*03:04': 'HLA-C*03:04',
 'HLA-C*04:01': 'HLA-C*04:01'}

# 1. Predict with NetMHC-3.4

In [10]:
from mhctools import NetMHC3

In [11]:
METHOD = 'netMHC-3.4'

def splitByLengthAndFilter(fasta_dict, min_length = 8):
    clean_fasta_dict = {k: v for k, v in fasta_dict.items() if not pd.isnull(v)}  #delete nan elements
    unique_lengths = list(set(map(len, list(clean_fasta_dict.values()))))
    unique_lengths = list(filter(lambda x: x>=min_length, unique_lengths ))
    per_length_dict_of_dicts = {target_length :{k: v for k, v in clean_fasta_dict.items() if len(v) == target_length} for target_length in unique_lengths}
    return per_length_dict_of_dicts

def getPredictor(allele_name):
    alleles = [allele_name]
    predictor = NetMHC3(
        alleles=alleles,
        program_name=METHOD)
    return predictor

def getSupportedAlleles():
    supported_alleles = NetMHC3._determine_supported_alleles(METHOD,"-A")
    return supported_alleles
    
    
def getSamplePredictionDf():
    protein_sequences = {
      "1L2Y": "NLYIQWLKD",
      "1L3Y": "ECDTINCER"
    }
    predictor = getPredictor("A*02:01")
    df = getPrediction(predictor=predictor, fasta_dict=protein_sequences, length=9)
    return df

def getPrediction(predictor, fasta_dict, length):
    predictions = predictor.predict_subsequences(fasta_dict,peptide_lengths=[length]).to_dataframe()
    return predictions
    
    
    
def fillDummyPrediction(fasta_dict, allele_name, length):
    sample_df = pd.DataFrame()
    sample_df['source_sequence_name'] = fasta_dict.keys()
    sample_df['offset'] = 0
    sample_df['peptide'] = fasta_dict.values()
    sample_df['allele'] = allele_name
    sample_df['score'] = np.nan
    sample_df['affinity'] = np.nan
    sample_df['percentile_rank'] = np.nan
    sample_df['prediction_method_name'] = np.nan
    sample_df['prediction_method_name'] = METHOD
    sample_df['length'] = length

    return sample_df
    
    
def predictSequences(dict_of_dicts, allele_name):
    supported_alleles = getSupportedAlleles()
    norm_allele_name = normalize_allele_name(TARGET_ALLELES[allele_name])# normalize allele name here 
    print(f"Predict binders for  dataset {allele_name} using alleles parameter {norm_allele_name}")
    if norm_allele_name in supported_alleles:
        predictor = getPredictor(norm_allele_name)
        binding_predictions = [getPrediction(predictor, fasta_dict, length)
                               for length, fasta_dict in dict_of_dicts.items()]
    else:
        print("!!!Unsupported allele, create dummy prediction!!!")
        binding_predictions = [fillDummyPrediction(fasta_dict,allele_name, length) for length, fasta_dict in dict_of_dicts.items()]
                    
    lengths = [len(prediction) for prediction in binding_predictions] 
    print(f"total processed peptides after filtering -  {sum(lengths)}")
    return binding_predictions

def predict_with_netmhc(per_allele_df):
    # make fasta-style dicts
    per_allele_fasta = {
        allele_name : {
            ID : PEPTIDE 
            for ID, PEPTIDE in zip('seq' + per_allele_df[allele_name].index.astype(str).values, per_allele_df[allele_name]['peptide'].values) 
        } 
        for allele_name in ALLELES 
    }
    
    # split by length
    per_allele_per_length_dict = dict() 
    for allele_name in ALLELES:
        fasta_dict = per_allele_fasta[allele_name]
        per_length_dict_of_dicts = splitByLengthAndFilter(fasta_dict)
        #print(f"allele {allele_name} fasta splited into {len(per_length_dict_of_dicts)} dicts")
        per_allele_per_length_dict[allele_name] = per_length_dict_of_dicts

    # predict
    per_allele_predictions = dict()
    for allele_name, per_length_dict in per_allele_per_length_dict.items():
        predictions = predictSequences(per_length_dict, allele_name)
        per_allele_predictions[allele_name] = predictions
    # join lengths together
    per_allele_total_predictions = dict()
    for allele_name, list_of_df in per_allele_predictions.items():
        df_result = pd.concat(list_of_df, ignore_index=True)
        per_allele_total_predictions[allele_name] = df_result
    return per_allele_total_predictions

In [12]:
getSamplePredictionDf()

,source_sequence_name,offset,peptide,allele,score,affinity,percentile_rank,prediction_method_name,length
0,1L3Y,0,ECDTINCER,HLA-A*02:01,0.053,28329.0,None,netMHC-3.4,9
1,1L2Y,0,NLYIQWLKD,HLA-A*02:01,0.056,27238.0,None,netMHC-3.4,9


In [13]:
for region in REGIONS:
    print(f"--------------------------------------------------------")
    print(f"REGION {region}")
    per_allele_mt_df = per_region_per_allele_mt_df[region]
    per_allele_wt_df = per_region_per_allele_wt_df[region]
    
    print(f"\t Mutated:")
    per_allele_mt_predictions = predict_with_netmhc(per_allele_mt_df)
    per_region_per_method_per_allele_mt_predictions[region][METHOD] = per_allele_mt_predictions

    print(f"\t Wild:")
    per_allele_wt_predictions = predict_with_netmhc(per_allele_wt_df)
    per_region_per_method_per_allele_wt_predictions[region][METHOD] = per_allele_wt_predictions    
    


--------------------------------------------------------
REGION 1
	 Mutated:
Predict binders for  dataset HLA-A*03:01 using alleles parameter HLA-A*03:01
total processed peptides after filtering -  5409
Predict binders for  dataset HLA-B*35:01 using alleles parameter HLA-B*35:01
total processed peptides after filtering -  5409
Predict binders for  dataset HLA-B*40:01 using alleles parameter HLA-B*40:01
total processed peptides after filtering -  5409
Predict binders for  dataset HLA-C*03:04 using alleles parameter HLA-C*03:04
!!!Unsupported allele, create dummy prediction!!!
total processed peptides after filtering -  5409
Predict binders for  dataset HLA-C*04:01 using alleles parameter HLA-C*04:01
total processed peptides after filtering -  5409
	 Wild:
Predict binders for  dataset HLA-A*03:01 using alleles parameter HLA-A*03:01
total processed peptides after filtering -  3751
Predict binders for  dataset HLA-B*35:01 using alleles parameter HLA-B*35:01
total processed peptides after f

# 2. Predict with NetMHC-4.0

In [244]:
from mhctools import NetMHC4
METHOD = 'netMHC-4.0'

In [245]:
def getPredictor(allele_name):
    alleles = [allele_name]
    predictor = NetMHC4(
        alleles=alleles,
        program_name=METHOD)
    return predictor
def getSupportedAlleles():
    supported_alleles = NetMHC4._determine_supported_alleles(METHOD,"-listMHC")
    return supported_alleles

In [246]:
getSamplePredictionDf()

,source_sequence_name,offset,peptide,allele,score,affinity,percentile_rank,prediction_method_name,length
0,1L3Y,0,ECDTINCER,HLA-A*02:01,0.026,37794.71,75.0,netMHC-4.0,9
1,1L2Y,0,NLYIQWLKD,HLA-A*02:01,0.092,18478.21,25.0,netMHC-4.0,9


In [247]:
for region in REGIONS:
    print(f"--------------------------------------------------------")
    print(f"REGION {region}")
    per_allele_mt_df = per_region_per_allele_mt_df[region]
    per_allele_wt_df = per_region_per_allele_wt_df[region]
    
    print(f"\t Mutated:")
    per_allele_mt_predictions = predict_with_netmhc(per_allele_mt_df)
    per_region_per_method_per_allele_mt_predictions[region][METHOD] = per_allele_mt_predictions

    print(f"\t Wild:")
    per_allele_wt_predictions = predict_with_netmhc(per_allele_wt_df)
    per_region_per_method_per_allele_wt_predictions[region][METHOD] = per_allele_wt_predictions    
    

--------------------------------------------------------
REGION 1
	 Mutated:
Predict binders for  dataset HLA-A*03:01 using alleles parameter HLA-A*03:01
total processed peptides after filtering -  5409
Predict binders for  dataset HLA-B*35:01 using alleles parameter HLA-B*35:01
total processed peptides after filtering -  5409
Predict binders for  dataset HLA-B*40:01 using alleles parameter HLA-B*40:01
total processed peptides after filtering -  5409
Predict binders for  dataset HLA-C*03:04 using alleles parameter HLA-C*03:04
!!!Unsupported allele, create dummy prediction!!!
total processed peptides after filtering -  5409
Predict binders for  dataset HLA-C*04:01 using alleles parameter HLA-C*04:01
total processed peptides after filtering -  5409
	 Wild:
Predict binders for  dataset HLA-A*03:01 using alleles parameter HLA-A*03:01
total processed peptides after filtering -  3751
Predict binders for  dataset HLA-B*35:01 using alleles parameter HLA-B*35:01
total processed peptides after f

# 3. Predict with NetMHCPan-4.1

In [131]:
from mhctools import NetMHCpan4

In [132]:
METHOD = 'netMHCpan-4.1'

In [145]:
def getPredictor(alleles):
    """
    mode - "elution_score" or "binding_affinity"
    """
    predictor_ba = NetMHCpan4(
        alleles=alleles,
        program_name=METHOD,
        mode="binding_affinity"
    )
    
    predictor_el = NetMHCpan4(
        alleles=alleles,
        program_name=METHOD,
        mode="elution_score"
    )
    return predictor_ba, predictor_el

def getSupportedAlleles():
    supported_alleles = NetMHCpan4._determine_supported_alleles(METHOD,"-listMHC")
    return supported_alleles

def reorderColumns(df):
    df = df[['source_sequence_name', 'offset', 'peptide', 'length', 'allele', 'score', 'affinity',
                      'percentile_rank_ba', 'percentile_rank_el', 'prediction_method_name' ]]
    return df

def getPrediction(predictor, fasta_dict, length):
    predictor_ba, predictor_el = predictor
    df_ba = predictor_ba.predict_subsequences(fasta_dict, peptide_lengths=[length]).to_dataframe()
    df_el = predictor_el.predict_subsequences(fasta_dict, peptide_lengths=[length]).to_dataframe()
    
    df_ba = df_ba.rename(columns={"percentile_rank": "percentile_rank_ba"})
    df_ba["percentile_rank_el"] = df_el["percentile_rank"]
    predictions = reorderColumns(df_ba)
    return predictions

def fillDummyPrediction(fasta_dict, allele_name, length):
    sample_df = pd.DataFrame()
    sample_df['source_sequence_name'] = fasta_dict.keys()
    sample_df['offset'] = 0
    sample_df['peptide'] = fasta_dict.values()
    sample_df['allele'] = allele_name
    sample_df['score'] = np.nan
    sample_df['affinity'] = np.nan
    sample_df['percentile_rank_el'] = np.nan
    sample_df['percentile_rank_ba'] = np.nan
    sample_df['prediction_method_name'] = METHOD
    sample_df['length'] = length
    return reorderColumns(df_ba)
    

In [146]:
getSamplePredictionDf()

,source_sequence_name,offset,peptide,length,allele,score,affinity,percentile_rank_ba,percentile_rank_el,prediction_method_name
0,1L3Y,0,ECDTINCER,9,HLA-A*02:01,0.000061,48.2,0.017415,48.2,netMHCpan-4.1
1,1L2Y,0,NLYIQWLKD,9,HLA-A*02:01,0.000089,43.0,0.074361,43.0,netMHCpan-4.1


In [147]:
for region in REGIONS:
    print(f"--------------------------------------------------------")
    print(f"REGION {region}")
    per_allele_mt_df = per_region_per_allele_mt_df[region]
    per_allele_wt_df = per_region_per_allele_wt_df[region]
    
    print(f"\t Mutated:")
    per_allele_mt_predictions = predict_with_netmhc(per_allele_mt_df)
    per_region_per_method_per_allele_mt_predictions[region][METHOD] = per_allele_mt_predictions

    print(f"\t Wild:")
    per_allele_wt_predictions = predict_with_netmhc(per_allele_wt_df)
    per_region_per_method_per_allele_wt_predictions[region][METHOD] = per_allele_wt_predictions    
    


--------------------------------------------------------
REGION 1
	 Mutated:
Predict binders for  dataset HLA-A*03:01 using alleles parameter HLA-A*03:01
total processed peptides after filtering -  5409
Predict binders for  dataset HLA-B*35:01 using alleles parameter HLA-B*35:01
total processed peptides after filtering -  5409
Predict binders for  dataset HLA-B*40:01 using alleles parameter HLA-B*40:01
total processed peptides after filtering -  5409
Predict binders for  dataset HLA-C*03:04 using alleles parameter HLA-C*03:04
total processed peptides after filtering -  5409
Predict binders for  dataset HLA-C*04:01 using alleles parameter HLA-C*04:01
total processed peptides after filtering -  5409
	 Wild:
Predict binders for  dataset HLA-A*03:01 using alleles parameter HLA-A*03:01
total processed peptides after filtering -  3751
Predict binders for  dataset HLA-B*35:01 using alleles parameter HLA-B*35:01
total processed peptides after filtering -  3751
Predict binders for  dataset HLA-

# 4. Join predictions

In [289]:
per_method_columns = {
    'netMHC-3.4': ['source_sequence_name' ,'score', 'affinity', 'percentile_rank'],
    'netMHC-4.0':['source_sequence_name', 'score', 'affinity', 'percentile_rank' ],
    'netMHCpan-4.1':['source_sequence_name', 'score', 'affinity', 'percentile_rank_ba', 'percentile_rank_el']
}



def rename_columns(df, method):
    leave_columns = ['source_sequence_name', 'peptide']
    old_columns = [col for col in per_method_columns[method] if col not in leave_columns]
    new_columns = [method + "_" + col for col in old_columns]
    df = df.rename(columns={old:new for old,new in zip(old_columns,new_columns)})
    return df
    

def getPredictionResult(total_results, region, allele_name, method):
    df = total_results[region][method][allele_name][per_method_columns[method]].copy()
    df = rename_columns(df, method)
    return df


In [290]:
def joinPredictionsToSource(per_region_per_method_per_allele_mt_predictions, per_region_per_method_per_allele_wt_predictions):
    per_region_per_allele_result = dict()
    for region in REGIONS:
        print(f"region {region}")
        per_region_per_allele_result[region] = dict()
        methods = list(per_region_per_method_per_allele_mt_predictions[region].keys())
        for allele_name in ALLELES:
            print(f"allele {allele_name}")
            df_source_wt = per_region_per_allele_wt_df[region][allele_name].copy()
            df_source_mt = per_region_per_allele_mt_df[region][allele_name].copy()
            
            for method in methods:
                print(f"method {method}")
                df_method_wt = getPredictionResult(per_region_per_method_per_allele_wt_predictions, region, allele_name, method)
                df_method_mt = getPredictionResult(per_region_per_method_per_allele_mt_predictions, region, allele_name, method)
                df_source_wt = df_source_wt.merge(df_method_wt, on=['source_sequence_name'], how='left')
                df_source_mt = df_source_mt.merge(df_method_mt, on=['source_sequence_name'], how='left')
            leave_columns = [ 'source_sequence_name', 'allele', 'length']
            df_source_wt = df_source_wt.rename(columns={col:  col + "_wt" for col in df_source_wt.columns if col not in leave_columns})
            df_source_mt = df_source_mt.rename(columns={col:  col + "_mt" for col in df_source_mt.columns if col not in leave_columns})
            # rename id back 

            
            df_result = df_source_mt.merge(df_source_wt, on=['source_sequence_name', 'allele', 'length'], how='left')
            per_region_per_allele_result[region][allele_name] = df_result
    return per_region_per_allele_result
                
                

              

In [291]:
per_region_per_allele_result = joinPredictionsToSource(per_region_per_method_per_allele_mt_predictions, per_region_per_method_per_allele_wt_predictions)

region 1
allele HLA-A*03:01
method netMHC-3.4
method netMHC-4.0
method netMHCpan-4.1
allele HLA-B*35:01
method netMHC-3.4
method netMHC-4.0
method netMHCpan-4.1
allele HLA-B*40:01
method netMHC-3.4
method netMHC-4.0
method netMHCpan-4.1
allele HLA-C*03:04
method netMHC-3.4
method netMHC-4.0
method netMHCpan-4.1
allele HLA-C*04:01
method netMHC-3.4
method netMHC-4.0
method netMHCpan-4.1
region 2
allele HLA-A*03:01
method netMHC-3.4
method netMHC-4.0
method netMHCpan-4.1
allele HLA-B*35:01
method netMHC-3.4
method netMHC-4.0
method netMHCpan-4.1
allele HLA-B*40:01
method netMHC-3.4
method netMHC-4.0
method netMHCpan-4.1
allele HLA-C*03:04
method netMHC-3.4
method netMHC-4.0
method netMHCpan-4.1
allele HLA-C*04:01
method netMHC-3.4
method netMHC-4.0
method netMHCpan-4.1
region 3
allele HLA-A*03:01
method netMHC-3.4
method netMHC-4.0
method netMHCpan-4.1
allele HLA-B*35:01
method netMHC-3.4
method netMHC-4.0
method netMHCpan-4.1
allele HLA-B*40:01
method netMHC-3.4
method netMHC-4.0
method

In [292]:
df_source_wt = per_region_per_allele_wt_df[region][allele_name]
df_source_wt

,HLA Allele,Peptide Length,peptide,source_sequence_name
14936,HLA-C*04:01,8,PPRTRNKF,seq14936
14939,HLA-C*04:01,8,SWPGTPVY,seq14939
14940,HLA-C*04:01,8,YVAVCRPL,seq14940
14941,HLA-C*04:01,8,RGLLSLAF,seq14941
14942,HLA-C*04:01,8,NaN,seq14942
...,...,...,...,...
65802,HLA-C*04:01,11,LRGTLQKDGWT,seq65802
65806,HLA-C*04:01,11,SSSSSSNNNNY,seq65806
65807,HLA-C*04:01,11,RSSSSSSSSNN,seq65807
65808,HLA-C*04:01,11,SSSSSSSSNNN,seq65808


In [293]:
df_source_mt = per_region_per_allele_mt_df[region][allele_name]
df_source_mt

,HLA Allele,Peptide Length,peptide,source_sequence_name
14936,HLA-C*04:01,8,TPRTRNKF,seq14936
14939,HLA-C*04:01,8,SWPGAPVY,seq14939
14940,HLA-C*04:01,8,YVAVCHPL,seq14940
14941,HLA-C*04:01,8,RELLSLAF,seq14941
14942,HLA-C*04:01,8,NQYASVAL,seq14942
...,...,...,...,...
65802,HLA-C*04:01,11,LHGTLQKDGWT,seq65802
65806,HLA-C*04:01,11,SSSSSSSNNNY,seq65806
65807,HLA-C*04:01,11,RSSSSSSSSSN,seq65807
65808,HLA-C*04:01,11,SSSSSSSSSNN,seq65808


In [295]:
per_region_per_allele_result[region][allele_name][['peptide_mt', 'peptide_wt', 'netMHCpan-4.1_percentile_rank_ba_mt','netMHCpan-4.1_percentile_rank_el_mt', 'netMHCpan-4.1_percentile_rank_ba_wt', 'netMHCpan-4.1_percentile_rank_el_wt']]

,peptide_mt,peptide_wt,netMHCpan-4.1_percentile_rank_ba_mt,netMHCpan-4.1_percentile_rank_el_mt,netMHCpan-4.1_percentile_rank_ba_wt,netMHCpan-4.1_percentile_rank_el_wt
0,TPRTRNKF,PPRTRNKF,0.041144,5.345,0.019331,23.625
1,SWPGAPVY,SWPGTPVY,0.074951,2.554,0.073769,2.246
2,YVAVCHPL,YVAVCRPL,0.116274,12.614,0.104894,18.508
3,RELLSLAF,RGLLSLAF,0.066993,14.260,0.068545,14.817
4,NQYASVAL,NaN,0.083399,4.082,NaN,NaN
...,...,...,...,...,...,...
4942,LHGTLQKDGWT,LRGTLQKDGWT,0.013230,90.000,0.014003,95.000
4943,SSSSSSSNNNY,SSSSSSNNNNY,0.024946,31.308,0.022948,33.273
4944,RSSSSSSSSSN,RSSSSSSSSNN,0.012414,90.000,0.012496,90.000
4945,SSSSSSSSSNN,SSSSSSSSNNN,0.009243,95.000,0.009001,95.000


In [281]:
per_region_per_allele_result[region][allele_name].columns

Index(['HLA Allele', 'Peptide Length', 'peptide_mt', 'source_sequence_name',
       'netMHC-3.4_score_mt', 'netMHC-3.4_affinity_mt',
       'netMHC-3.4_percentile_rank_mt', 'netMHC-4.0_score_mt',
       'netMHC-4.0_affinity_mt', 'netMHC-4.0_percentile_rank_mt',
       'netMHCpan-4.1_score_mt', 'netMHCpan-4.1_affinity_mt',
       'netMHCpan-4.1_percentile_rank_ba_mt',
       'netMHCpan-4.1_percentile_rank_el_mt', 'peptide_wt',
       'netMHC-3.4_score_wt', 'netMHC-3.4_affinity_wt',
       'netMHC-3.4_percentile_rank_wt', 'netMHC-4.0_score_wt',
       'netMHC-4.0_affinity_wt', 'netMHC-4.0_percentile_rank_wt',
       'netMHCpan-4.1_score_wt', 'netMHCpan-4.1_affinity_wt',
       'netMHCpan-4.1_percentile_rank_ba_wt',
       'netMHCpan-4.1_percentile_rank_el_wt'],
      dtype='object')

# 5. For now just write to files

In [302]:
METHOD = 'NetMHC'
DATA_PATH = "/home/denklewer/Data/gbm122/"

TARGET_PATH_TO_PREDICTIONS = DATA_PATH + f'predictions/{METHOD}/'

import os
if not os.path.exists(TARGET_PATH_TO_PREDICTIONS):
    os.makedirs(TARGET_PATH_TO_PREDICTIONS)

In [303]:
# for decoy_key in DECOY_PREFIXES:

for region in REGIONS:
    for allele_name in ALLELES:
        total_df = per_region_per_allele_result[region][allele_name]
        total_df.to_csv(f"{TARGET_PATH_TO_PREDICTIONS}{METHOD}_predictions_reg_{region}_allele_{allele_name}", , index=False, na_rep="NaN")